# Level of Effort

:::{div}
:class: colab-badge
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blinkletter/UsingPsi4v2/blob/main/ch2C-level-of-effort.ipynb)
:::


In [ ]:
#| label: setup-2C
#| tags: [remove-cell, remove-stdout]

####################################################
### Import Libraries and set up global variables ###
####################################################

from pathlib import Path
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Psi4 cannot be installed with pip. On Colab we first install conda
# (condacolab), which RESTARTS the session. After the restart, run
# this cell again and Psi4 will be installed with conda.
try:
    import psi4
except ImportError:
    if IN_COLAB:
        !pip -q install condacolab
        import condacolab
        condacolab.install()
        !mamba install -q -y -c conda-forge psi4
    import psi4

try:
    import py3Dmol
except ImportError:
    !pip -q install py3Dmol
    import py3Dmol

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Style sheets and helper functions from the book's GitHub repository
if IN_COLAB:
    if not Path("styles/tufte.mplstyle").exists():
        !wget -q -N -P styles https://raw.githubusercontent.com/blinkletter/UsingPsi4v2/main/styles/tufte.mplstyle
        !wget -q -N https://raw.githubusercontent.com/blinkletter/UsingPsi4v2/main/psi4_helpers.py
    plt.style.use("styles/tufte.mplstyle")
    nproc = 2          # Colab gives you 2 cores
else:
    plt.style.use("styles/tufte.mplstyle")
    nproc = 4          # house limit: 4 threads

import psi4_helpers as ph

# All Psi4 .out and .log files go into the output folder
output_dir = Path("output/ch2/2C-level-of-effort")
output_dir.mkdir(exist_ok=True)

# Resources: house limits are 4 threads and 4 GB
psi4.set_num_threads(nproc)
_ = psi4.set_memory("2 GB")

# 3D pictures: False = static images (as in the book); True = rotatable py3Dmol viewers
INTERACTIVE = True

Water is so **small** that it needs little effort to produce the calculated results. However, we can **increase** this effort in hopes of achieving greater **accuracy**. Less intense methods make simplifying assumptions or leave out important math that affects the final energy. Are you **treating** the electron pair in an orbital as an electric field (HF and DFT) or are you **keeping track** of the positions of each electron as they try to **avoid** one another within the orbital (MP2, CCSD and others). How much **work** do you want your computer to do?

```{note} The Invisible Code
:class: dropdown
There is a **block** of code **at the top** if each notebook that is **invisible** in this web book. When **running** the notebook in *Colab* or on your own computer, that cell will be visible and needs to be the **first** block **executed**. This setup block at the top of each notebook **sets the resources** that *Psi4* may use. It will sense the environment and, if you are using *Colab*, the code will **install** required *Python* tools. On your own computer you will have set up your *Python* environment in advance according to the [setup instructions](ch1-setup.md).

In this notebook, the hidden code also decalres the **destination** for all output file (`output/ch2/2C-level-of-effort`), sets the **number of processors** to use (2 for *Colab*, 4 for my own computer), and sets **flags** that affect many different cells in the notebook. Here, the flag `INTERACTIVE = False` will make all moelcular images render as **still images**. If you set `INTERACTIVE = True`, then those images will be 3-dimensional **interactive objects** that you can spin around and observe at any angle. The interactive images can be very, **very large** when they include molecular orbitals and other objects. The web book is executed with `INTERACTIVE = False` so that I **don't break** the cloud provider that is serving these pages. You can set it to `True` when running your calculation in *Colab* or on your own machine.
```
No pain, no gain. Every calculation is a choice of **method** (how the electrons interact) and **basis set** (the functions used to build the orbitals). Better choices cost more computer time. In this notebook we optimize water at low, medium and high levels of effort and compare the structures with experiment.

## What You Will Learn
- The main families of **methods**: Hartree–Fock (HF), density functional theory (DFT), MP2 and coupled cluster (CCSD(T)).
- The main families of **basis sets**: minimal (STO-3G), Pople (6-31G*), and correlation-consistent (cc-pVDZ, cc-pVTZ, aug-cc-pVTZ).
- How accuracy and cost change together, and why "more expensive" does not always mean "better" in every property.

The experimental equilibrium structure of water is $r_{\mathrm{OH}} = 0.9578$ Å and $\angle\mathrm{HOH} = 104.48°$. {cite:p}`HOY19791`